In [3]:
import numpy as np

# Variable size of submatrix (layer = size)
size = 3

# Input matrix
matrix = np.array([
    [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    [0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0]
])

rows, cols = matrix.shape
num_layers = rows // size

# Replace bypass (all-zero) submatrices with identity matrix
for i in range(0, rows, size):
    for j in range(0, cols, size):
        sub = matrix[i:i+size, j:j+size]
        if np.all(sub == 0):
            matrix[i:i+size, j:j+size] = np.eye(sub.shape[0], sub.shape[1])

def prev_one_in_row(row_ones, col):
    """Return the column index of the previous 1 in the row before 'col'.
    If none exists (col is leftmost 1), wrap around to the last 1 in the row."""
    before = [x for x in row_ones if x < col]
    if before:
        return max(before)   # nearest 1 to the left
    else:
        return max(row_ones) # wrap: last 1 in the row

# Generate edgelist:
# For each column, for each layer, find the column index of the previous 1
# in the same row as the current column's 1. Store as packed hex nibbles.
# Nibble[0] (MSN) = layer 0, Nibble[1] = layer 1, Nibble[2] = layer 2, ...
print("EdgeList (previous 1 per layer, packed as hex):")
for col in range(cols):
    nibbles = []
    for layer in range(num_layers):
        row_start = layer * size
        # Find which row in this layer has a 1 for this column
        rel_rows = np.where(matrix[row_start:row_start+size, col] == 1)[0]
        if len(rel_rows) == 0:
            nibbles.append(0)
            continue
        abs_row = row_start + rel_rows[0]
        # Get all columns in this row that have a 1
        row_ones = np.where(matrix[abs_row, :] == 1)[0].tolist()
        # Find the previous 1 in the row (with circular wrap)
        prev_idx = prev_one_in_row(row_ones, col)
        nibbles.append(prev_idx)
    # Pack nibbles into a single integer (MSN = layer 0)
    val = 0
    for nib in nibbles:
        val = (val << 4) | nib
    bits = num_layers * 4
    print(f"        edgelist[{col:2d}] = {bits}'h{val:0{num_layers}x};")

EdgeList (previous 1 per layer, packed as hex):
        edgelist[ 0] = 12'ha99;
        edgelist[ 1] = 12'hbaa;
        edgelist[ 2] = 12'h9bb;
        edgelist[ 3] = 12'h111;
        edgelist[ 4] = 12'h222;
        edgelist[ 5] = 12'h000;
        edgelist[ 6] = 12'h354;
        edgelist[ 7] = 12'h435;
        edgelist[ 8] = 12'h543;
        edgelist[ 9] = 12'h767;
        edgelist[10] = 12'h878;
        edgelist[11] = 12'h686;


In [4]:
import numpy as np

# Redefine original matrix to detect bypass (since cell 1 modifies it)
matrix_orig = np.array([
    [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    [0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0]
])

size = 3
num_layers = matrix_orig.shape[0] // size
cols = matrix_orig.shape[1]

def prev_one_in_row(row_ones, col):
    """Return the column index of the previous 1 in the row before 'col'.
    If none exists (col is leftmost 1), wrap around to the last 1 in the row."""
    before = [x for x in row_ones if x < col]
    if before:
        return max(before)   # nearest 1 to the left
    else:
        return max(row_ones) # wrap: last 1 in the row

print("Bypass List (previous 1 to bypass):")
for j in range(cols):
    bypass_val = 15 # Default 4'hF
    for l in range(num_layers):
        # Check if the submatrix for this column in this layer is all 0s
        sub_col = matrix_orig[l*size:(l+1)*size, j]
        if np.all(sub_col == 0):
            # Determine which row within this block *would* have had the 1
            # Based on the original MATLAB construction, bypass submatrices are replaced by Identity
            # so the 1 would be at relative row: (j // size) doesn't work. The identity matrix 
            # replacement is: matrix[i:i+size, j:j+size] = eye(size). 
            # Thus, for column j, the 1 is placed at relative row (j % size).
            abs_row = (l * size) + (j % size)
            row_ones = np.where(matrix_orig[abs_row, :] == 1)[0].tolist()
            if row_ones:
                # Get the previous 1 in that row relative to j
                bypass_val = prev_one_in_row(row_ones, j)
            break # A column is bypassed in at most one layer
            
    print(f"        bypass_list[{j:2d}] = 4'h{bypass_val:X};")


Bypass List (previous 1 to bypass):
        bypass_list[ 0] = 4'h9;
        bypass_list[ 1] = 4'hA;
        bypass_list[ 2] = 4'hB;
        bypass_list[ 3] = 4'hF;
        bypass_list[ 4] = 4'hF;
        bypass_list[ 5] = 4'hF;
        bypass_list[ 6] = 4'h3;
        bypass_list[ 7] = 4'h4;
        bypass_list[ 8] = 4'h5;
        bypass_list[ 9] = 4'hF;
        bypass_list[10] = 4'hF;
        bypass_list[11] = 4'hF;
